<a href="https://colab.research.google.com/github/Aneekcodes07/postman-hackathon-project-yousum/blob/main/ML_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

# pandas as pd
# Used for handling tabular data (CSV, Excel)
# You’ll use it to load and manipulate dataset
# numpy as np
# Used for numerical operations (arrays, math)
# Not heavily used here but standard in ML projects
# re
# Regular expressions → used for text cleaning
# Example: remove special characters
# nltk
# Natural Language Toolkit → used for text preprocessing
# Important for removing stopwords
# matplotlib.pyplot as plt
# Used for basic graphs
# seaborn as sns
# Used for better-looking statistical plots

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import resample

# train_test_split
# Splits data into training + testing
# WHY? → To check if model works on unseen data
# TfidfVectorizer
# Converts text → numbers
# WHY? → ML models cannot understand text
# MultinomialNB
# Naive Bayes algorithm for text classification
# Best choice for your problem
# classification_report
# Gives:
# Precision
# Recall
# F1-score
# confusion_matrix
# Shows:
# Correct predictions
# Wrong predictions
# resample
# Used to handle class imbalance
# Very important in your project

In [3]:
nltk.download('stopwords')
from nltk.corpus import stopwords

# Explanation:
# Downloads stopwords like:
# the, is, and, etc.

# WHY remove them?
# Because they don’t add meaning in classification

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
df = pd.read_csv("/content/sample_data/fake_job_postings.csv", engine='python',on_bad_lines='skip')
# Why this works:
# engine='python'
# 👉 More flexible parser (handles messy data better)
# on_bad_lines='skip'
# 👉 Skips corrupted rows instead of crashing
df.head(5)

# The dataset contains raw textual data, which may include inconsistent
#  formatting such as unclosed quotes or special characters, causing parsing errors.
#  We handled this using a robust parsing method and skipped corrupted rows.”


FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/fake_job_postings.csv'

In [ ]:
print("Dataset Shape:", df.shape)

# 👉 Helps understand dataset size
# Shows:
# Rows
# Columns


In [ ]:
print(df['fraudulent'].value_counts())
# Counts:
# Real jobs (0)
# Fake jobs (1)

# Helps detect class imbalance

In [ ]:
# Set modern style
sns.set_theme(style="whitegrid")

# Create figure
plt.figure(figsize=(7,5))

# Custom colors (professional palette)
colors = ["#4CAF50", "#F44336"]  # Green = Real, Red = Fraud

ax = sns.countplot(
    x='fraudulent',
    data=df,
    palette=colors
)

# Title & labels (clean font styling)
plt.title("Real vs Fraudulent Job Postings", fontsize=16, fontweight='bold')
plt.xlabel("Class (0 = Real, 1 = Fraud)", fontsize=12)
plt.ylabel("Number of Job Postings", fontsize=12)

# Customize ticks
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# Add values on top of bars
for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width()/2., p.get_height() + 50),
        ha='center',
        fontsize=11,
        fontweight='bold',
        color='black'
    )

# Remove top/right borders for clean look
sns.despine()

plt.tight_layout()
plt.show()
# 🔍 Explanation:
# countplot → shows number of real vs fake jobs
# Helps visualize imbalance

# Viva point:

# “We performed EDA to understand class distribution”

In [ ]:
df['text'] = (
    df['title'].fillna('') + " " +
    df['company_profile'].fillna('') + " " +
    df['description'].fillna('') + " " +
    df['requirements'].fillna('')
)
df

# What This Does (Simple)

# It creates one single column called text
# This column contains all important information combined
# Example

# Before:
# title	description	requirements
# Data Scientist	Work with ML	Python, ML

# After:
# "Data Scientist Work with ML Python, ML"

# ❓ WHY Do We Combine?
# 👉 Machine learning models:
# ❌ Cannot take multiple text columns directly
# ✅ Need one input feature
# 🎯 Key Reason:

# We want the model to learn from all text together

# 🔍 Line-by-Line Explanation
# df['text'] = (...)
# 👉 Creating a new column called text
# df['title']
# 👉 Takes job title text

# fillna('')

# 👉 Replaces missing values with empty string

# ❗ WHY?

# Avoid errors like:

# TypeError: can only concatenate str (not "NaN")
# " "
# 👉 Adds space between columns

# ❗ Without this:
# "DataScientistPythonML" ❌
### +

# 👉 Concatenates (joins) strings
# ⚠️ VERY IMPORTANT (Common Mistake)

# 👉 If you don’t use fillna('')
# ➡️ Your code will crash ❌\
# porfessional
# text_columns = ['title', 'company_profile', 'description', 'requirements']

# df['text'] = df[text_columns].fillna('').agg(' '.join, axis=1)

In [ ]:
# Load stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove HTML tags (if any)
    text = re.sub(r'<.*?>', '', text)

    # 3. Remove special characters & numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # 4. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Apply cleaning
df['text'] = df['text'].apply(clean_text)


# 1. text.lower()

#  Converts everything to lowercase
# WHY?
# "Job" and "job" should be treated same
# 🔹 2. Remove HTML tags
# re.sub(r'<.*?>', '', text)

#  Removes things like:

# <p>Apply now</p>
# 🔹 3. Remove special characters & numbers
# re.sub(r'[^a-z\s]', '', text)

#  Removes:

# Numbers: 123
# Symbols: @#$%
# WHY?
# They don’t help in understanding meaning
# 🔹 4. Remove extra spaces
# re.sub(r'\s+', ' ', text).strip()

# Converts:

# "job   posting   data"

# "job posting data"
# 🔹 5. Remove Stopwords
# words = [word for word in words if word not in stop_words]

# Removes:

# the, is, and, in, etc.
# WHY?
# These words don’t add meaning
# 📌 Example (Before vs After)
# BEFORE:
# "Apply NOW!!! Work from home @ company123!!!"
# AFTER:
# "apply work home company"

8: HANDLE CLASS IMBALANCE

In [ ]:
from sklearn.utils import resample

# Separate classes
df_majority = df[df.fraudulent == 0]
df_minority = df[df.fraudulent == 1]

# Upsample minority class
df_minority_upsampled = resample(
    df_minority,
    replace=True,                        # allow duplication
    n_samples=len(df_majority),          # match majority count
    random_state=42
)

# Combine both classes
df_balanced = pd.concat([df_majority, df_minority_upsampled])

# Shuffle dataset
df_balanced = df_balanced.sample(frac=1, random_state=42)

print(df_balanced['fraudulent'].value_counts())

# What is Class Imbalance?

# 👉 Your dataset looks like this:

# Class	Count
# Real (0)	~7961
# Fraud (1)	~387

# 👉 Problem:

# Model learns mostly real jobs
# Ignores fraud ❌
# ⚠️ Why It’s Dangerous

# 👉 Model may give:

# 95% accuracy
# BUT fails to detect fraud

# 👉 That’s why:

# Accuracy ≠ Good model

# 🔥 Solution Used: Oversampling

# 👉 We increase minority class (fraud)

# 🔍 Line-by-Line Explanation
# 🔹 Separate Classes
# df_majority = df[df.fraudulent == 0]
# df_minority = df[df.fraudulent == 1]

# 👉 Split dataset into:

# Real jobs
# Fraud jobs
# 🔹 Resample (Upsampling)
# resample(df_minority, replace=True, n_samples=len(df_majority))

# 👉 Key parameters:

# replace=True
# Allows duplication of rows
# Needed because minority is small
# n_samples=len(df_majority)
# Make fraud = real jobs

# 👉 Balanced dataset

# 🔹 Combine
# pd.concat([...])

# 👉 Merge both classes

# 🔹 Shuffle
# sample(frac=1)

# 👉 Randomize data

# WHY?

# Prevent model bias
# 📊 Before vs After
# Stage	Real	Fraud
# Before	7961	387
# After	7961	7961
# 🎯 Why This Improves Recall

# 👉 Model now:

# Sees more fraud examples
# Learns fraud patterns better

# 👉 Result:

# Higher Recall 🔥

9. Train test split

In [ ]:

# Features (input) and target (output)
X = df_balanced['text']
y = df_balanced['fraudulent']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% test, 80% train
    random_state=42,      # for reproducibility
    stratify=y            # maintain class balance
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

🧠 What is Train-Test Split?

👉 We divide data into:

Part	Purpose
Training Data	Used to train model
Testing Data	Used to evaluate model
🎯 Why Do We Split?

👉 If you train and test on same data:

Model will memorize ❌
Not generalize

👉 So we:

Train on one part, test on unseen data

🔍 Line-by-Line Explanation
🔹 Features & Target
X = df_balanced['text']
y = df_balanced['fraudulent']
X → input text
y → output label (0 or 1)
🔹 train_test_split(...)
test_size=0.2

👉 20% data for testing

Data	%
Train	80%
Test	20%
random_state=42

👉 Fixes randomness

WHY?

Same split every time
Reproducible results
stratify=y 🔥 (VERY IMPORTANT)

👉 Keeps class distribution same in both sets

Example:

Dataset	Real	Fraud
Train	50%	50%
Test	50%	50%
⚠️ What Happens Without stratify

👉 You might get:

Train: mostly real
Test: mostly fraud

👉 Model becomes unreliable ❌

📊 Example Output
Training size: (12737,)
Testing size: (3185,)

👉 Means:

Training data = 80%
Testing data = 20%

Step 1: TF-IDF1

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=15000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
# Create model
model = MultinomialNB()

# Train model
model.fit(X_train_tfidf, y_train)

11.preditcion

🧠 What is Prediction?

👉 After training, the model:

Takes new (test) data
Uses learned patterns
Predicts output (0 or 1)
🔍 What Happens Internally

For each job posting:

TF-IDF converts text → numbers
Naive Bayes calculates probabilities
Chooses class with highest probability
💡 Example

Input:

"earn money fast no experience required"

Model thinks:

Class	Probability
Real (0)	0.2
Fraud (1)	0.8

👉 Output:

1  # Fraud
🔎 Understanding model.predict()
y_pred = model.predict(X_test_tfidf)

👉 Output:

Array of predictions
Example:
[0, 0, 1, 0, 1, 1, 0]

In [ ]:
# Make predictions on test data
y_pred = model.predict(X_test_tfidf)

# Show first few predictions
print(y_pred[:10])

# Compare actual and predicted
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print(comparison.tail(9))

12: MODEL EVALUATION

In [ ]:
# Classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Plot confusion matrix
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

13 VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt

counts = df['fraudulent'].value_counts()
labels = ['Real Jobs', 'Fraud Jobs']
colors = ['#2ecc71', '#e74c3c']

plt.figure(figsize=(6,6))
plt.pie(counts, labels=labels, autopct='%1.1f%%',
        colors=colors, startangle=140,
        wedgeprops={'width':0.2})

plt.title("Job Posting Distribution", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Create text length column
df['text_length'] = df['text'].apply(lambda x: len(x.split()))

# Now plot
plt.figure(figsize=(8,5))

sns.boxplot(
    x='fraudulent',
    y='text_length',
    data=df,
    palette=['#3498db', '#e67e22']
)

plt.title("Text Length Distribution by Class", fontsize=14, fontweight='bold')
plt.xlabel("Class (0 = Real, 1 = Fraud)")
plt.ylabel("Number of Words")

plt.show()

What is this graph?

This is a Box Plot (Box-and-Whisker Plot)
It shows the distribution of text length for:

0 → Real jobs
1 → Fraud jobs
What each part means
🔹 The Box

Represents middle 50% of data (IQR)

Bottom of box → 25th percentile
Top of box → 75th percentile
🔹 The Line inside box

This is the Median (middle value)

In your graph:

Real jobs → median ≈ 200 words
Fraud jobs → median ≈ 150–170 words
🔹 The Whiskers

Show the normal range of data

🔹 The Dots

These are outliers

Meaning:

Extremely long job descriptions
Unusual cases
 What YOUR graph tells
 Key Insights
1. Real jobs are slightly longer
Median higher for class 0
More detailed descriptions
2. Fraud jobs are shorter (on average)
Lower median
Less detailed
3. Real jobs have more extreme outliers
Some very long descriptions (1000+ words)
 Why this is useful

Helps you understand:

Writing pattern differences
Fraud jobs tend to be:
Shorter
Less detailed

TOP WORDS (HORIZONTAL BAR)

In [ ]:
from collections import Counter

words = " ".join(df['text']).split()
common_words = Counter(words).most_common(15)

w = [i[0] for i in common_words]
c = [i[1] for i in common_words]

plt.figure(figsize=(10,5))
sns.barplot(x=c, y=w, palette='mako')

plt.title("Top 15 Most Common Words", fontsize=14, fontweight='bold')
plt.xlabel("Frequency")

plt.show()

WORD CLOUD

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='viridis'
).generate(" ".join(df['text']))

plt.figure(figsize=(10,5))
plt.imshow(wordcloud)
plt.axis('off')

plt.title("Word Cloud of Job Postings", fontsize=14, fontweight='bold')
plt.show()

TF-IDF FEATURE IMPORTANCE

In [ ]:

feature_names = vectorizer.get_feature_names_out()
scores = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

top_indices = scores.argsort()[-15:]
top_features = feature_names[top_indices]
top_scores = scores[top_indices]

plt.figure(figsize=(10,5))
sns.barplot(x=top_scores, y=top_features, palette='flare')

plt.title("Top TF-IDF Features", fontsize=14, fontweight='bold')
plt.xlabel("Importance Score")

plt.show()

CONFUSION MATRIX

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='coolwarm',
    linewidths=1,
    linecolor='black'
)

plt.title("Confusion Matrix (Model Performance)", fontsize=14, fontweight='bold')
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

METRICS VISUALIZATION (BAR CHART)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score

# Calculate metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Define labels and values
metrics = ['Precision', 'Recall', 'F1 Score']
values = [precision, recall, f1]

# Plot
plt.figure(figsize=(6,4))
sns.set_theme(style="whitegrid")

colors = ['#66BB6A', '#42A5F5', '#FFA726']

sns.barplot(x=metrics, y=values, palette=colors)

plt.ylim(0,1)
plt.title("Model Performance Metrics", fontsize=13, fontweight='bold')

# Add values on bars
for i, v in enumerate(values):
    plt.text(i, v+0.02, f"{v:.2f}", ha='center')

sns.despine()
plt.tight_layout()
plt.show()

REAL vs FRAUD WORD COMPARISON

In [ ]:
real_words = " ".join(df[df.fraudulent==0]['text']).split()
fraud_words = " ".join(df[df.fraudulent==1]['text']).split()

real_common = Counter(real_words).most_common(10)
fraud_common = Counter(fraud_words).most_common(10)

fig, axes = plt.subplots(1,2, figsize=(12,5))

sns.barplot(x=[i[1] for i in real_common],
            y=[i[0] for i in real_common],
            ax=axes[0], palette='Blues')
axes[0].set_title("Real Jobs Words")

sns.barplot(x=[i[1] for i in fraud_common],
            y=[i[0] for i in fraud_common],
            ax=axes[1], palette='Reds')
axes[1].set_title("Fraud Jobs Words")

plt.tight_layout()
plt.show()